In [ ]:
# Imports
import scipy.io as sio
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import os
import copy

# Set working directory
if os.getcwd() != '/home/jovyan/Semantic-Embedding-Evolution/notebooks':
    try:
        os.chdir('Semantic-Embedding-Evolution/notebooks')
    except FileNotFoundError:
        # Fallback for local testing if path differs
        pass

# Data setup
%run ../setup/setup.py

# Static variables
seed = 42
years = range(1990, 2017)
vocab_size = 20936
embeddings_file = "../data/emb_static.mat"
word_id_file = "../data/wordIDHash.csv"
train_path = "../data/pmi_"

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load static embeddings
static_embeddings = sio.loadmat(embeddings_file)["emb"]
static_embeddings = torch.from_numpy(static_embeddings).float()

# Word to ID mapping
df_word_id = pd.read_csv(word_id_file, header=None, names=["id", "word", "count"])
wordlist = [f"missing_word_idx_{i}" for i in range(vocab_size)]
for _, row in df_word_id.iterrows():
    wordlist[int(row["id"])] = str(row["word"])
word2Id = {word: i for i, word in enumerate(wordlist)}

# Dataset Class
class PMIDataset(Dataset):
    def __init__(self, time_steps, train_path, vocab_size):
        self.time_steps = time_steps
        self.train_path = train_path
        self.vocab_size = vocab_size

    def __len__(self):
        return len(self.time_steps)

    def __getitem__(self, idx):
        t_idx = self.time_steps[idx]
        file_path = f"{train_path}{t_idx}.mat"

        pmi = sio.loadmat(file_path)["pmi"]
        pmi = torch.from_numpy(pmi.toarray()).float()

        return t_idx, pmi

# Model Definition
class DynamicTransformerModel(nn.Module):
    def __init__(self, n_years, initial_emb, n_head=4, n_layers=2, dropout=0.1, dim_feedforward=200):
        super().__init__()
        self.n_years = n_years
        vocab_size, dim = initial_emb.shape
        
        # Static embedding (learnable initialization)
        self.static_emb = nn.Parameter(initial_emb.clone())
        
        # Positional encoding for years
        self.year_pos_emb = nn.Parameter(torch.zeros(n_years, 1, dim))
        nn.init.normal_(self.year_pos_emb, mean=0, std=0.1)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=n_head, dim_feedforward=dim_feedforward, dropout=dropout)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # Output heads
        self.head_U = nn.Linear(dim, dim)
        self.head_V = nn.Linear(dim, dim)

    def forward(self):
        # Input: (S, N, E) -> (n_years, vocab_size, dim)
        
        # Expand static embeddings: (N, E) -> (1, N, E) -> (S, N, E)
        x = self.static_emb.unsqueeze(0).expand(self.n_years, -1, -1)
        
        # Add positional embeddings: (S, 1, E) -> (S, N, E)
        x = x + self.year_pos_emb
        
        # Transformer
        encoded = self.transformer(x)
        
        # Heads
        U = self.head_U(encoded)
        V = self.head_V(encoded)
        
        return U, V

    @torch.no_grad()
    def get_embeddings(self):
        U, _ = self.forward()
        return U.detach()

    @torch.no_grad()
    def get_drift(self, word_idx):
        U = self.get_embeddings()
        start_vec = U[0, word_idx]
        end_vec = U[-1, word_idx]

        cos_sim = torch.cosine_similarity(start_vec, end_vec, dim=0).item()
        drift_norm = torch.norm(start_vec - end_vec).item()

        return drift_norm, cos_sim

# Training Function
def train_model(model, dataset, years_range, n_epochs=25, lr=1e-3, weight_decay=1e-5, 
                tau=0.005, gam=0.5, emph=5000, device=device, verbose=True):
    
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Map year index in dataset to model index (0 to n_years-1)
    # dataset indices are absolute (0 to 26 for 1990-2016)
    # model indices are relative to the training period
    
    # We need to know which dataset indices correspond to the model's years
    # years_range should be a list of indices into the full dataset
    
    model.to(device)
    model.train()
    
    for epoch in range(n_epochs):
        indices = torch.randperm(len(years_range))
        epoch_loss = 0
        
        # Forward pass for all years in the model
        # Note: In the original transformer notebook, we did forward pass inside the loop?
        # Actually, for transformer, we need to pass all years at once to get temporal context if we want self-attention across time?
        # Wait, the transformer takes (S, N, E). If we split training, the "S" changes.
        # If we pre-train on 1990-2008, S=19.
        # If we fine-tune on 2009-2016, S=8? Or do we keep the history?
        # "Fine-tuning" usually implies continuing training.
        # If we just train on 2009-2016 with a new model instance initialized from the old one, 
        # the transformer won't see 1990-2008 data during forward pass.
        # This might be what is intended: "fine-tune transformer with data after YYYY".
        # However, for a temporal model, context matters.
        # But if the model architecture is fixed to N years, we can't easily change N.
        # Let's assume we define a NEW model for the fine-tuning period, but initialize its weights 
        # (static_emb) from the pre-trained model?
        # Or do we use the SAME model (S=27) but only compute loss on the later years?
        # "fine-tune transformer with data after YYYY" implies we only use data after YYYY for the loss.
        # If we use the full model (1990-2016) but only optimize loss for 2009-2016, that's one way.
        # But that requires loading all data.
        
        # Let's interpret "fine-tuning" as:
        # 1. Train Model A on 1990-2008.
        # 2. Create Model B for 2009-2016. Initialize Model B's static embeddings from Model A's output at 2008?
        #    Or just initialize Model B's parameters from Model A?
        #    The positional embeddings would be different (year 0 in B is 2009).
        
        # Alternative interpretation:
        # Use the SAME model structure (covering 1990-2016).
        # Phase 1: Train on 1990-2008 (mask loss for 2009-2016).
        # Phase 2: Train on 2009-2016 (mask loss for 1990-2008, or maybe keep it? No, "fine-tune with data after").
        
        # Let's go with the "Masked Loss" approach on a full model first, as it keeps the architecture consistent.
        # But wait, if we only train on 1990-2008, the model learns nothing about 2009-2016 positions.
        
        # Let's try the "Transfer Learning" approach:
        # Model 1: 1990-2008.
        # Model 2: 2009-2016.
        # Initialize Model 2 using Model 1's final state?
        # The static_emb is the "base" meaning.
        # If we take Model 1's `static_emb` and use it as init for Model 2, that's transfer.
        
        # Let's stick to the "Masked Loss" approach on the FULL timeframe for simplicity of implementation 
        # and to allow the transformer to potentially see the whole sequence if we were to feed it all.
        # BUT, if we only feed 1990-2008 data, we can't compute loss for 2009-2016.
        
        # Let's define the training function to accept a list of `active_years` indices.
        # The model will always be defined for `n_years` (e.g. 27 or subset).
        
        # Actually, if we want to answer "does fine-tuning modify semantic meaning", we should probably:
        # 1. Pre-train a model on 1990-2008. (Model covers 19 years).
        # 2. Fine-tune: Create a model for 2009-2016 (8 years). Initialize it?
        #    Or maybe the user implies we have a model that CAN handle the new data.
        
        # Let's assume the "Full Model" covers 1990-2016.
        # Pre-training: Optimize loss only for t < split.
        # Fine-tuning: Optimize loss only for t >= split.
        
        # This seems most consistent with "fine-tuning" in NLP (e.g. BERT): you train on a subset of tasks/data.
        
        # So, `train_model` will take `active_indices` which are the years to compute loss for.
        
        torch.cuda.empty_cache()
        
        # We need to run forward pass.
        # If the model is defined for ALL years, we run forward for all years.
        # But we only compute loss for `active_indices`.
        
        U_all, V_all = model()
        
        for y_idx in indices: # y_idx is index into years_range
            dataset_idx = years_range[y_idx] # The actual year index (0-26)
            
            # We need to map dataset_idx to the model's output index.
            # If model covers 1990-2016, model_idx = dataset_idx.
            # If model covers 1990-2008, model_idx = dataset_idx.
            # If model covers 2009-2016, model_idx = dataset_idx - split_idx.
            
            # To keep it simple, let's assume the model passed to this function 
            # corresponds exactly to the `years_range` passed.
            # i.e. if years_range is 2009-2016, the model has 8 years.
            
            model_idx = y_idx # Since we shuffle indices, this is wrong.
            # indices is a permutation of 0..len(years_range)-1
            # So y_idx is the index in the *subset*.
            
            # Wait, `indices` is `torch.randperm(len(years_range))`.
            # So `y_idx` IS the index in the subset (0 to subset_len-1).
            
            # Get embeddings for this year
            U_t = U_all[y_idx]
            V_t = V_all[y_idx]
            
            # Load PMI
            _, pmi = dataset[dataset_idx]
            
            # Calculate reconstruction loss in chunks
            rec_loss = 0
            chunk_size = 2048
            
            for i in range(0, vocab_size, chunk_size):
                end = min(i + chunk_size, vocab_size)
                pmi_chunk = pmi[i:end].to(device)
                U_chunk = U_t[i:end]
                pred_chunk = torch.matmul(U_chunk, V_t.t())
                
                mask_chunk = (pmi_chunk > 0).float()
                weight_chunk = mask_chunk * emph + (1 - mask_chunk)
                rec_loss += torch.sum(weight_chunk * torch.square(pmi_chunk - pred_chunk))
                
                del pmi_chunk, pred_chunk, mask_chunk, weight_chunk
            
            # Alignment loss
            alignment_loss = gam * torch.sum(torch.square(U_t - V_t))
            
            # Temporal loss
            temporal_loss = 0
            if y_idx > 0:
                temporal_loss = tau * torch.sum(torch.square(U_t - U_all[y_idx - 1]))
                
            loss = rec_loss + alignment_loss + temporal_loss
            
            optimizer.zero_grad()
            loss.backward(retain_graph=True) # Retain graph because U_all is used across loop? 
            # No, U_all is computed once per epoch? 
            # If we optimize per year, we should probably re-compute U_all or detach?
            # In the original notebook, we did:
            # for y_idx in indices:
            #    U_all, V_all = model()
            #    ...
            #    loss.backward()
            #    optimizer.step()
            
            # So we re-compute forward pass every step. That's safer.
            
        # Correct loop structure:
        for step_i in range(len(years_range)):
            # Pick a random year from the range
            # Actually, iterating through a permutation is better.
            pass
            
    # Let's rewrite the loop to match the original notebook's logic exactly
    # but adapted for subsets.
    
    return model

def train_epoch(model, dataset, years_indices, optimizer, tau, gam, emph, device):
    model.train()
    epoch_loss = 0
    
    # Shuffle the years we are training on
    permuted_indices = torch.randperm(len(years_indices))
    
    for i in permuted_indices:
        # The index in the model's timeframe (0 to model_years-1)
        model_y_idx = i.item()
        
        # The index in the full dataset (0 to 26)
        dataset_y_idx = years_indices[model_y_idx]
        
        # Forward pass
        U_all, V_all = model()
        
        U_t = U_all[model_y_idx]
        V_t = V_all[model_y_idx]
        
        # Load data
        _, pmi = dataset[dataset_y_idx]
        
        # Loss calculation (same as before)
        rec_loss = 0
        chunk_size = 2048
        for j in range(0, vocab_size, chunk_size):
            end = min(j + chunk_size, vocab_size)
            pmi_chunk = pmi[j:end].to(device)
            U_chunk = U_t[j:end]
            pred_chunk = torch.matmul(U_chunk, V_t.t())
            mask_chunk = (pmi_chunk > 0).float()
            weight_chunk = mask_chunk * emph + (1 - mask_chunk)
            rec_loss += torch.sum(weight_chunk * torch.square(pmi_chunk - pred_chunk))
            del pmi_chunk, pred_chunk, mask_chunk, weight_chunk
            
        alignment_loss = gam * torch.sum(torch.square(U_t - V_t))
        
        temporal_loss = 0
        if model_y_idx > 0:
            temporal_loss = tau * torch.sum(torch.square(U_t - U_all[model_y_idx - 1]))
            
        loss = rec_loss + alignment_loss + temporal_loss
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(years_indices)



In [ ]:
# Training Helper Function
def train_model_on_years(model, dataset, years_indices, n_epochs=25, lr=1e-3, weight_decay=1e-5, 
                         tau=0.005, gam=0.5, emph=5000, device=device):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.to(device)
    
    history = []
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        
        # Shuffle indices for this epoch
        permuted_indices = torch.randperm(len(years_indices))
        
        for i in permuted_indices:
            # model_y_idx: index in the model's timeframe (0 to model.n_years-1)
            model_y_idx = i.item()
            
            # dataset_y_idx: index in the full dataset (0 to 26)
            dataset_y_idx = years_indices[model_y_idx]
            
            # Forward pass
            U_all, V_all = model()
            
            U_t = U_all[model_y_idx]
            V_t = V_all[model_y_idx]
            
            # Load data
            _, pmi = dataset[dataset_y_idx]
            
            # Loss calculation
            rec_loss = 0
            chunk_size = 2048
            for j in range(0, vocab_size, chunk_size):
                end = min(j + chunk_size, vocab_size)
                pmi_chunk = pmi[j:end].to(device)
                U_chunk = U_t[j:end]
                pred_chunk = torch.matmul(U_chunk, V_t.t())
                mask_chunk = (pmi_chunk > 0).float()
                weight_chunk = mask_chunk * emph + (1 - mask_chunk)
                rec_loss += torch.sum(weight_chunk * torch.square(pmi_chunk - pred_chunk))
                del pmi_chunk, pred_chunk, mask_chunk, weight_chunk
                
            alignment_loss = gam * torch.sum(torch.square(U_t - V_t))
            
            temporal_loss = 0
            if model_y_idx > 0:
                temporal_loss = tau * torch.sum(torch.square(U_t - U_all[model_y_idx - 1]))
                
            loss = rec_loss + alignment_loss + temporal_loss
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / len(years_indices)
        history.append(avg_loss)
        print(f"Epoch {epoch+1}/{n_epochs} | Loss: {avg_loss:.2f}")
        
    return history

# Dataset instance
full_dataset = PMIDataset(range(len(years)), train_path, vocab_size)


In [ ]:
# Experiment 1: Full Training (Baseline)
# Train on all years (1990-2016) at once.

print("--- Experiment 1: Full Training (Baseline) ---")
full_years_indices = list(range(len(years))) # 0 to 26
model_baseline = DynamicTransformerModel(len(years), static_embeddings, n_head=4, n_layers=2, dropout=0.1)

# Train
history_baseline = train_model_on_years(model_baseline, full_dataset, full_years_indices, n_epochs=25)

# Save baseline embeddings for comparison
baseline_embeddings = model_baseline.get_embeddings().cpu()
print("Baseline training complete.")

In [ ]:
# Experiment 2: Pre-training and Fine-tuning
# Split year: 2010 (Index 20)
# Pre-train: 1990-2009 (Indices 0-19)
# Fine-tune: 2010-2016 (Indices 20-26)

print("\n--- Experiment 2: Pre-training and Fine-tuning ---")

split_year = 2010
split_idx = split_year - 1990
pre_train_indices = list(range(split_idx))
fine_tune_indices = list(range(split_idx, len(years)))

print(f"Split Year: {split_year}")
print(f"Pre-training years: {years[0]}-{years[split_idx-1]} ({len(pre_train_indices)} years)")
print(f"Fine-tuning years: {years[split_idx]}-{years[-1]} ({len(fine_tune_indices)} years)")

# Model for Experiment 2
# We use the SAME model architecture covering all years, but we only train on subsets.
# This allows the model to have parameters for all years, but we only optimize specific ones.
# Wait, if we don't train the later years during pre-training, their embeddings will just be 
# random/static initialization + position encoding.
# And if we don't train earlier years during fine-tuning, they will stay as they were (mostly).

model_finetune = DynamicTransformerModel(len(years), static_embeddings, n_head=4, n_layers=2, dropout=0.1)

# Phase 1: Pre-training
print("Phase 1: Pre-training...")
history_pre = train_model_on_years(model_finetune, full_dataset, pre_train_indices, n_epochs=25)

# Save pre-trained state
pretrained_embeddings = model_finetune.get_embeddings().cpu()

# Phase 2: Fine-tuning
print("Phase 2: Fine-tuning...")
# We continue training the SAME model, but now iterating over the fine-tuning years.
history_fine = train_model_on_years(model_finetune, full_dataset, fine_tune_indices, n_epochs=25)

# Save fine-tuned embeddings
finetuned_embeddings = model_finetune.get_embeddings().cpu()
print("Fine-tuning complete.")

In [ ]:
# Experiment 3: How much fine-tuning is needed?
# We will fine-tune for different numbers of epochs and measure the change.

print("\n--- Experiment 3: Fine-tuning Epoch Analysis ---")

# Reset model to pre-trained state
model_epoch_analysis = DynamicTransformerModel(len(years), static_embeddings, n_head=4, n_layers=2, dropout=0.1)
# Load state from Phase 1 of Exp 2 (we need to have saved the state_dict, but we didn't. 
# Let's just re-train or copy weights if possible. 
# Since we didn't save state_dict, we'll just re-run pre-training quickly or reuse the previous object if we hadn't overwritten it.
# Actually, `model_finetune` is already fine-tuned. We can't go back.
# We should have saved the state dict.

# Let's re-run pre-training for this experiment to be clean.
print("Re-running pre-training for Experiment 3...")
train_model_on_years(model_epoch_analysis, full_dataset, pre_train_indices, n_epochs=25, verbose=False)
pretrained_state = copy.deepcopy(model_epoch_analysis.state_dict())

fine_tune_epochs_list = [1, 5, 10, 25, 50]
drift_results = []

target_word = "amazon"
word_idx = word2Id[target_word]

for ft_epochs in fine_tune_epochs_list:
    print(f"Fine-tuning for {ft_epochs} epochs...")
    
    # Reset to pre-trained state
    model_epoch_analysis.load_state_dict(copy.deepcopy(pretrained_state))
    model_epoch_analysis.to(device) # Ensure on device
    
    # Fine-tune
    train_model_on_years(model_epoch_analysis, full_dataset, fine_tune_indices, n_epochs=ft_epochs, verbose=False)
    
    # Measure drift for target word in the fine-tuning period
    # We compare embedding at split_year (2010) vs end year (2016)
    # Note: split_idx corresponds to 2010.
    
    embeddings = model_epoch_analysis.get_embeddings().cpu()
    
    # Drift from start of fine-tuning to end
    vec_start = embeddings[split_idx, word_idx]
    vec_end = embeddings[-1, word_idx]
    
    drift_norm = torch.norm(vec_start - vec_end).item()
    cos_sim = torch.cosine_similarity(vec_start.unsqueeze(0), vec_end.unsqueeze(0)).item()
    
    drift_results.append({
        "Epochs": ft_epochs,
        "Drift L2": drift_norm,
        "Cosine Sim": cos_sim
    })

df_drift = pd.DataFrame(drift_results)
print(df_drift)

In [ ]:
# Visualization and Comparison

def plot_comparison(word, baseline_embs, finetuned_embs, word2Id, years, split_year):
    word_idx = word2Id[word]
    
    # Extract trajectories
    traj_base = baseline_embs[:, word_idx, :].numpy()
    traj_ft = finetuned_embs[:, word_idx, :].numpy()
    
    # Combine for TSNE
    combined = np.vstack([traj_base, traj_ft])
    
    tsne = TSNE(n_components=2, random_state=42, perplexity=5)
    Z = tsne.fit_transform(combined)
    
    Z_base = Z[:len(years)]
    Z_ft = Z[len(years):]
    
    plt.figure(figsize=(12, 6))
    
    # Plot Baseline
    plt.plot(Z_base[:, 0], Z_base[:, 1], 'o-', label='Baseline (Full Training)', alpha=0.5)
    for i, year in enumerate(years):
        if year % 5 == 0 or year == years[0] or year == years[-1]:
            plt.text(Z_base[i, 0], Z_base[i, 1], str(year), fontsize=8)
            
    # Plot Fine-tuned
    plt.plot(Z_ft[:, 0], Z_ft[:, 1], 's--', label='Fine-tuned', alpha=0.5)
    for i, year in enumerate(years):
        if year >= split_year: # Only label fine-tuned years relevantly
             if year % 2 == 0:
                plt.text(Z_ft[i, 0], Z_ft[i, 1], str(year), fontsize=8, color='red')

    plt.title(f"Embedding Trajectory Comparison for '{word}'")
    plt.legend()
    plt.grid(True)
    plt.show()

# Compare specific words
words_to_compare = ["amazon", "apple", "obama", "trump"]

for w in words_to_compare:
    if w in word2Id:
        plot_comparison(w, baseline_embeddings, finetuned_embeddings, word2Id, years, split_year)
    else:
        print(f"Word '{w}' not in vocabulary.")

# Quantitative Comparison
# Calculate average cosine similarity between baseline and fine-tuned embeddings for all words in the fine-tuning period
print("\n--- Quantitative Comparison (2010-2016) ---")
sims = []
for y_idx in range(split_idx, len(years)):
    # Compare embeddings for this year
    # Shape: (Vocab, Dim)
    base_y = baseline_embeddings[y_idx]
    ft_y = finetuned_embeddings[y_idx]
    
    # Cosine similarity per word
    cos = torch.nn.functional.cosine_similarity(base_y, ft_y, dim=1)
    avg_sim = cos.mean().item()
    sims.append(avg_sim)
    print(f"Year {years[y_idx]}: Avg Cosine Similarity = {avg_sim:.4f}")

print(f"Overall Average Similarity in Fine-tuning Period: {np.mean(sims):.4f}")